In [ ]:
#imports de bibliotecas 

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time
from datetime import datetime



In [ ]:
# CONFIGURAÇÕES INICIAIS

# URL da página inicial 
BASE_URL = "https://pt.wikipedia.org/wiki/Luiz_In%C3%A1cio_Lula_da_Silva"
BASE_DOMAIN = "https://pt.wikipedia.org"

# Caminho onde os HTMLs serão salvos
RAW_PATH = "data/raw_html"
# Caminho onde os CSVs serão salvos
OUTPUT_PATH = "data/wikipedia"

# Cria as pastas caso não existam
os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
# FUNÇÃO PARA BAIXAR HTML
def get_html(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.text
    except:
        return None

In [ ]:
def get_html(url):
    """
    Realiza uma requisição HTTP e retorna o HTML da página.
    Inclui headers para simular um navegador e tratamento de erros.
    """
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        print("Status code:", response.status_code)

        # Verifica se a requisição foi bem-sucedida
        if response.status_code == 200:
            return response.text
        else:
            return None

    except Exception as e:
        print("Erro:", e)
        return None

In [ ]:

# Verifica se o HTML foi carregado corretamente
main_html = get_html(BASE_URL)

if main_html:
    with open(f"{RAW_PATH}/main.html", "w", encoding="utf-8") as f:
        f.write(main_html)
    print("Página principal salva com sucesso!")
else:
    print("Falha ao baixar a página principal")

Status code: 200
Página principal salva com sucesso!


In [ ]:
# EXTRAÇÃO DE LINKS INTERNOS (/wiki/)
if main_html:
    soup = BeautifulSoup(main_html, "html.parser")

    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        
        # Filtra apenas links internos da Wikipedia
        # Remove páginas especiais e âncoras
        if href.startswith("/wiki/") and not any(x in href for x in [
            "/wiki/Especial:",
            "/wiki/Ajuda:",
            "/wiki/Ficheiro:",
            "/wiki/Categoria:",
            "#"
        ]):
            full_link = BASE_DOMAIN + href.split("#")[0]
            links.add(full_link)

    print(f"Total de links filtrados: {len(links)}")
else:
    print("HTML não carregado — abortando extração")

Total de links filtrados: 14


In [26]:
for i, link in enumerate(links):
    html = get_html(link)
    
    if html:
        with open(f"{RAW_PATH}/page_{i}.html", "w", encoding="utf-8") as f:
            f.write(html)
    
    time.sleep(1)

Status code: 404
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200
Status code: 200


In [ ]:
# Limita para evitar excesso de requisições
links = list(links)[:50]
print(f"Links após corte: {len(links)}")

Links após corte: 14


In [ ]:
# DOWNLOAD DAS PÁGINAS INTERNAS
for i, link in enumerate(links):
    html = get_html(link)
    
    if html:
        # Salva cada página com nome único
        with open(f"{RAW_PATH}/page_{i}.html", "w", encoding="utf-8") as f:
            f.write(html)
        print(f"[{i}] OK")
    else:
        print(f"[{i}] ERRO: {link}")
    # Pausa para evitar bloqueio
    time.sleep(1)

Status code: 404
[0] ERRO: https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:Isen%C3%A7%C3%A3o_de_responsabilidade_geral
Status code: 200
[1] OK
Status code: 200
[2] OK
Status code: 200
[3] OK
Status code: 200
[4] OK
Status code: 200
[5] OK
Status code: 200
[6] OK
Status code: 200
[7] OK
Status code: 200
[8] OK
Status code: 200
[9] OK
Status code: 200
[10] OK
Status code: 200
[11] OK
Status code: 200
[12] OK
Status code: 200
[13] OK


In [33]:
def parse_html(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser")

        # título
        title_tag = soup.find("h1")
        title = title_tag.text.strip() if title_tag else None

        # primeiro parágrafo válido
        paragraphs = soup.find_all("p")
        first_paragraph = None

        for p in paragraphs:
            if p.text.strip():
                first_paragraph = p.text.strip()
                break

        # imagens
        images = []
        for img in soup.find_all("img"):
            src = img.get("src")
            if src:
                images.append("https:" + src)

        # links internos
        internal_links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            
            if href.startswith("/wiki/") and ":" not in href:
                internal_links.append(BASE_DOMAIN + href)

        return title, first_paragraph, images, internal_links

    except Exception as e:
        print(f"Erro ao processar {file_path}: {e}")
        return None, None, [], []

In [ ]:
# PROCESSAMENTO DOS HTMLs
data_pages = []
data_images = []

files = os.listdir(RAW_PATH)

print(f"Total de arquivos: {len(files)}")
# Percorre todos os arquivos HTML salvos
for file in files:
    file_path = os.path.join(RAW_PATH, file)

    title, paragraph, images, links = parse_html(file_path)
    timestamp = datetime.now()
    # Armazena dados da página
    data_pages.append({
        "title": title,
        "first_paragraph": paragraph,
        "links": links,
        "timestamp": timestamp
    })
    # Armazena dados das imagens
    for img in images:
        data_images.append({
            "image_url": img,
            "page_title": title,
            "timestamp": timestamp
        })

Total de arquivos: 15


In [ ]:
# CRIAÇÃO DOS DATAFRAMES
df_pages = pd.DataFrame(data_pages)
df_images = pd.DataFrame(data_images)

df_pages.head()

,title,first_paragraph,links,timestamp
0,Luiz Inácio Lula da Silva,Luiz Inácio Lula da Silva (nascido Luiz Inácio...,[https://pt.wikipedia.org/wiki/Luiz_In%C3%A1ci...,2026-04-04 00:08:06.386065
1,Luiz Inácio Lula da Silva,Luiz Inácio Lula da Silva (nascido Luiz Inácio...,[https://pt.wikipedia.org/wiki/Luiz_In%C3%A1ci...,2026-04-04 00:08:06.950258
2,Portal:Índice,Portais destacados · Diretório de portais · !T...,[],2026-04-04 00:08:07.970713
3,Portal:Eventos atuais,Sobre o portal,[],2026-04-04 00:08:08.051363
4,Discussão:Luiz Inácio Lula da Silva,O principal objectivo do WikiProjecto Biografi...,[https://pt.wikipedia.org/wiki/Luiz_In%C3%A1ci...,2026-04-04 00:08:08.079394


In [ ]:
# EXPORTAÇÃO PARA CSV
df_pages.to_csv(f"{OUTPUT_PATH}/pages.csv", index=False)
df_images.to_csv(f"{OUTPUT_PATH}/images.csv", index=False)

print("CSV gerados com sucesso!")

CSV gerados com sucesso!


In [ ]:
# VALIDAÇÃO FINAL
print("Resumo:")

print("Páginas coletadas:", len(df_pages))
print("Imagens coletadas:", len(df_images))
print(df_pages["title"].iloc[0])

Resumo:
Páginas coletadas: 15
Imagens coletadas: 868
Luiz Inácio Lula da Silva
